In [2]:
from pydantic import BaseModel, Field
from typing import Optional, Literal, List, Dict
from datetime import date


#Management quality (MQ) classes

class Mq_indicator(BaseModel):
    indicator_id : str
    indicator_desc : str
    score : Literal["Yes", "No", "Not Applicable"] = None # "not applicable" is only for Q12 which is only applicable for certain sectors

class Level(BaseModel):   
    level : Literal["L0", "L1", "L2", "L3", "L4", "L5"] = None #might be easier way to do this
    mq_indicators : List[Mq_indicator]

class Mq_score(BaseModel):
    levels : List[Level]

class Mq_summary(BaseModel):
    mq_publication_date : Optional[date] = None    
    mq_assesment_date : Optional[date] = None
    overall_level : float
    performance_compared_previous_year : Optional[Literal["unchanged", "new"]] = None

class Management_quality(BaseModel):
    mq_score : Optional[Mq_score] = None
    mq_summary : Optional[Mq_summary] = None

#Carbon performance (CP) classes

#carbon perforamance intensity classes
class Yearly_carbon_intensity_values(BaseModel):
    year : Optional[int] = None
    value : Optional[str] = None

class Carbon_intensity(BaseModel):
    cutoff_year : Optional[float] = None
    cp_measurement_unit : Optional[str] = None
    history_to_projection_cutoff_year : Optional[float] = None
    yearly_carbon_intensity : list[Yearly_carbon_intensity_values]

#carbon performance alignment classes
class Current_cp_alignment(BaseModel):
    years_with_target : Optional[str] = None
    current_cp_alignment_2025 : Optional[str] = None
    current_cp_alignment_2027 : Optional[str] = None
    current_cp_alignment_2035 : Optional[str] = None
    current_cp_alignment_2050 : Optional[str] = None

class Previous_cp_alignment(BaseModel):
    prev_years_with_target : Optional[str] = None
    prev_cp_alignment_2025 : Optional[str] = None
    prev_cp_alignment_2027 : Optional[str] = None
    prev_cp_alignment_2035 : Optional[str] = None
    prev_cp_alignment_2050 : Optional[str] = None

class Cp_alignment(BaseModel):
    assumptions : Optional[str] = None  
    latest_cp_alignment : Optional[Current_cp_alignment] = None
    previous_cp_alignment : Optional[Previous_cp_alignment] = None
    carbon_intensity : Optional[Carbon_intensity] = None  #MAKE SURE THIS IS REFLECTED IN FUNCTIONS!!

 
 #sector benchmark classes
class Benchmark(BaseModel):
    release_date : Optional[date] = None
    senario_name : Optional[str] = None
    region : Optional[str] = None
    unit : Optional[str] = None
    carbon_intesity : List[Yearly_carbon_intensity_values]

class Cp_sector_benchmarks(BaseModel):
    benchmarks : List[Benchmark]


#overall carbon perforamnce class
class Carbon_performance_summary(BaseModel):
    cp_publication_date : Optional[date] = None  
    cp_assesment_date : Optional[date] = None
    benchmark_id : Optional[str] = None

class Carbon_performance(BaseModel):
    carbon_performance_summary : Optional[Carbon_performance_summary] = None #may need to use List[]
    latest_cp_alignment : Optional[Current_cp_alignment] = None
    previous_cp_alignment : Optional[Previous_cp_alignment] = None
    cp_sector_benchmarks : Optional[Benchmark] = None

#extension classes

#Summary classes

class Company_info(BaseModel):
    geography : str
    geography_code : str
    sector: str
    CA100_focus_company : bool
    l_m_class : Literal["large", "medium", "small"]
    isins : str
    sedol : Optional[str] = None

class Company(BaseModel):   
    company_name : str
    company_info : Company_info
    carbon_performance : Carbon_performance
    management_quality : Management_quality



In [18]:
# make sure to add regional output for companies with regional data
import os
import pandas as pd

#load and clean corporate assessments data
df_assessments = pd.read_csv("C:/Users/User/OneDrive/Documents/GitHub/DS205/problem-set-1-ds205-2025-felix-brown/tpi-corprate-api/data/TPI sector data - All sectors - 20022025/Company_Latest_Assessments_5.0.csv")
assesment_date_cols = ["MQ Publication Date", "MQ Assessment Date", "CP Publication Date", "CP Assessment Date"]
df_assessments[assesment_date_cols] = df_assessments[assesment_date_cols].apply(pd.to_datetime, format='%d/%m/%Y')

#load and clean sector benchmark data
df_benchmarks = pd.read_csv("C:/Users/User/OneDrive/Documents/GitHub/DS205/problem-set-1-ds205-2025-felix-brown/tpi-corprate-api/data/TPI sector data - All sectors - 20022025/Sector_Benchmarks_20022025.csv")
df_benchmarks["Release date"] = pd.to_datetime(df_benchmarks["Release date"], format = "%d/%m/%Y")



#general purpose functions

#function that extracts company assesments data
def get_company_data(company):
    data = df_assessments[df_assessments["Company Name"] == company]
    return data


#function that raises http exeptions
def raise_http_exception(data, company, variable):
    if data is None or data.empty:  
        raise HTTPException(status_code=404, 
                            detail=f"There is no data on company: {company} 's variable: {variable}")
    

#function that raises http exception if there is no sector data




# add function documentation

# management quality functions
def create_mq_indicator(mq_indicator, data): 
    mq_indicator_col = data.filter(like = mq_indicator)

    column_name = mq_indicator_col.columns[0]

    mq_indicator_dict = {
        "indicator_id": mq_indicator,
        "indicator_desc": column_name.split("|")[1],
        "score": mq_indicator_col.iloc[0]
    }
    
    return mq_indicator_dict

def create_level(level, data):
    level_cols = data.filter(like = level).columns
    mq_indicator_names = [col.split("|")[0] for col in level_cols]
    mq_indicator_list = [create_mq_indicator(mq_indicator, data) for mq_indicator in mq_indicator_names]

    level_dict = {
        "level": level,  #maybe just make this be the int part
        "mq_indicators": mq_indicator_list
    }

    return level_dict

def create_mq_score(data):
    levels = ["L0", "L1", "L2", "L3", "L4", "L5"]
    level_list = [create_level(level, data) for level in levels]

    mq_score_dict = {"levels": level_list}

    return mq_score_dict

def create_mq_summary(data):
    mq_summary_dict = {
        "mq_publication_date": data["MQ Publication Date"].iloc[0],
        "mq_assesment_date": data["MQ Assessment Date"].iloc[0],
        "overall_level": data["Overall Level"].iloc[0],
        "performance_compared_previous_year": data["Performance Compared to Previous Year"].iloc[0]
    }

    return mq_summary_dict

def create_management_quality(data):
    mq_dictionary = {
        "mq_score" : create_mq_score(data),
        "mq_summary" : create_mq_summary(data)
    }

    return mq_dictionary


#carbon intensity functions

def create_yearly_carbon_intensity(data, year):
    year_cp_dictionary = {
        "year" : year,                     #does it make sense to do this rather than year : value?
        "value" : data[year].iloc[0]       #could be that column names are #year eg. #2014
    }
    
    return year_cp_dictionary


def create_carbon_intensity(data):
    
    #commentasdffffffffffffffffff
    valid_years = data[[
        col for col in data.columns if col.isdigit() and 2013 <= int(col) <= 2050
        ]]

    carbon_intensity_dict = {
        "cutoff_year" : data["Cutoff Year"].iloc[0],
        "cp_measurement_unit" : data["CP Measurement Unit"].iloc[0],
        "history_to_projection_cutoff_year" : data["History to Projection Cutoff Year"].iloc[0],
        "yearly_carbon_intensity" : 
            [create_yearly_carbon_intensity(data, year) for year in valid_years]   #is this correct indentation?
    } 

    return carbon_intensity_dict

def create_current_cp_alignment(data):
    latest_cp_alignment_dict = {
        "current_years_with_target" : data["Years with Target"].iloc[0],
        "current_cp_alignment_2025" : data["Carbon Performance Alignment 2025"].iloc[0],
        "current_cp_alignment_2027" : data["Carbon Performance Alignment 2027"].iloc[0],
        "current_cp_alignment_2035" : data["Carbon Performance Alignment 2035"].iloc[0],
        "current_cp_alignment_2050" : data["Carbon Performance Alignment 2050"].iloc[0]
    }

    return latest_cp_alignment_dict

def create_prev_cp_alignment(data):
    prev_cp_alignment_dict = {
        "prev_years_with_target" : data["Previous Years with Target"].iloc[0],
        "prev_cp_alignment_2025" : data["Previous Carbon Performance Alignment 2025"].iloc[0],
        "prev_cp_alignment_2027" : data["Previous Carbon Performance Alignment 2027"].iloc[0],
        "prev_cp_alignment_2035" : data["Previous Carbon Performance Alignment 2035"].iloc[0],
        "prev_cp_alignment_2050" : data["Previous Carbon Performance Alignment 2050"].iloc[0]
    }

    return prev_cp_alignment_dict

def create_cp_alignment(data):
    cp_alignment_dict = {
        "assumptions" : data["Assumptions"].iloc[0],
        "latest_cp_alignment" : create_current_cp_alignment(data),
        "previous_cp_alignment" : create_prev_cp_alignment(data)
    }

    return cp_alignment_dict

def create_cp_summary(data):
    cp_summary_dict = {
        "cp_publication_date" : data["CP Publication Date"].iloc[0],
        "cp_assesment_date" : data["CP Assessment Date"].iloc[0],
        "benchmark_id" : data["Benchmark ID"].iloc[0]
    }

    return cp_summary_dict

#sector benchmark functions

#create benchmark
def create_benchmark(company):
    
    company_data = get_company_data(company)
    
    benchmark_data = (df_benchmarks["Benchmark ID"] == company_data["Benchmark ID"])

    benchmark_intensity_list = [create_yearly_carbon_intensity(benchmark_data, year) for year in range(2013, 2051)]
    benchmark_intensity_list = Yearly_carbon_intensity_values(**benchmark_intensity_list)

    benchmark_dict = {
        "release_date" : benchmark_data["Release date"].iloc[0],
        "senario_name" : benchmark_data["Scenario Name"].iloc[0],
        "region" : benchmark_data["Region"].iloc[0],
        "unit" : benchmark_data["Unit"].iloc[0],
        "carbon_intensity" : benchmark_intensity_list
    }

    return benchmark_dict

def create_cp_sector_benchmarks(company):
    sector_benchmarks_df = df_benchmarks[df_benchmarks["Sector"] == get_company_data(company)["Sector"].iloc[0]]
    
    sector_benchmarks_list =  []

    for benchmark in sector_benchmarks_df["Benchmark ID"]:

        benchmark_df = sector_benchmarks_list["Benchmark ID" == benchmark]

        benchmark_intensity_list_2 = [create_yearly_carbon_intensity(benchmark_df, year) for year in range(2013, 2051)]
        benchmark_intensity_list_2 = Yearly_carbon_intensity_values(**benchmark_intensity_list_2)

        benchmark_dict_2 = {
        "release_date" : benchmark_df["Release date"].iloc[0],
        "senario_name" : benchmark_df["Scenario Name"].iloc[0],
        "region" : benchmark_df["Region"].iloc[0],
        "unit" : benchmark_df["Unit"].iloc[0],
        "carbon_intensity" : benchmark_intensity_list_2
    }
        sector_benchmarks_list.append(benchmark_dict_2)

    return sector_benchmarks_list


def create_carbon_performance(data):
    carbon_performance_dict = {
        "carbon_performance_summary" : create_cp_summary(data),
        "carbon_performance_alignment" : create_cp_alignment(data),
        "previous_cp_alignment" : create_prev_cp_alignment(data),
        "cp_sector_benchmarks" : create_cp_sector_benchmarks(data)
    }

    return carbon_performance_dict


# company summary  functions

def create_company_info(data):
    company_info_dict = {
        "geography" : data["Geography"].iloc[0],
        "geography_code" : data["Geography Code"].iloc[0],
        "sector" : data["Sector"].iloc[0],
        "CA100_focus_company" : data["CA100 Focus Company"].iloc[0],
        "l_m_class" : data["L/M Class"].iloc[0],
        "isins" : data["ISINs"].iloc[0],
        "sedol" : data["SEDOL"].iloc[0]
    }

    return company_info_dict

def create_company(data):
    company_dict = {
        "company_name" : data["Company Name"].iloc[0],
        "company_info" : create_company_info(data),
        "carbon_performance" : create_carbon_performance(data),
        "management_quality" : create_management_quality(data)
    }

    return company_dict


In [ ]:
def create_mq_indicator(mq_indicator, data): 
    mq_indicator_col = data.filter(like = mq_indicator)

    column_name = mq_indicator_col.columns[0]

    mq_indicator_dict = {
        "indicator_id": mq_indicator,
        "indicator_desc": column_name.split("|")[1],
        "score": mq_indicator_col.iloc[0, 0]
    }
    
    return mq_indicator_dict


def create_level(level, data):
    level_cols = data.filter(like = level).columns
    mq_indicator_names = [col.split("|")[0] for col in level_cols]
    mq_indicator_list = [create_mq_indicator(mq_indicator, data) for mq_indicator in mq_indicator_names]

    level_dict = {
        "level": level,  #maybe just make this be the int part
        "mq_indicators": mq_indicator_list
    }

    return level_dict

def create_mq_score(data):
    levels = ["L0", "L1", "L2", "L3", "L4", "L5"]
    level_list = [create_level(level, data) for level in levels]

    mq_score_dict = {"levels": level_list}

    return mq_score_dict

data = get_company_data("3M")

def create_mq_summary(data):
    mq_summary_dict = {
        "mq_publication_date": data["MQ Publication Date"].iloc[0],
        "mq_assesment_date": data["MQ Assessment Date"].iloc[0],
        "overall_level": data["Level"].iloc[0],
        "performance_compared_previous_year": data["Performance Compared to Previous Year"].iloc[0]
    }

    return mq_summary_dict

create_mq_summary(data)



{'mq_publication_date': Timestamp('2024-12-01 00:00:00'),
 'mq_assesment_date': Timestamp('2024-04-30 00:00:00'),
 'overall_level': 3.0,
 'performance_compared_previous_year': 'down'}